In [48]:
# %pip install numpy pandas scikit-learn keras tensorflow matplotlib seaborn scapy xgboost

In [49]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [50]:
import warnings
warnings.filterwarnings("ignore")

In [51]:
processed_filepath = './datasets/processed/'
data = pd.read_csv(processed_filepath + 'Thursday.csv')
df = data.copy()


# df = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')

In [52]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,389,113095465,48,24,9668,10012,403,0,201.416667,203.548293,...,32,203985.500,5.758373e+05,1629110,379,13800000.0,4.277541e+06,16500000,6737603,0
1,389,113473706,68,40,11364,12718,403,0,167.117647,171.919413,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4.229413e+06,16500000,6945512,0
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.333,1.170000e+07,20400000,6,24400000.0,2.430000e+07,60100000,5702188,0
3,443,60261928,9,7,2330,4221,1093,0,258.888889,409.702161,...,20,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,0
4,53,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.000000e+00,0,0,0


In [53]:
print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [54]:
X = df.drop('Label', axis=1).values
y = df['Label'].values

In [55]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [56]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Obliczanie wag klasowych
# from sklearn.utils.class_weight import compute_class_weight
# class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# class_weights = dict(enumerate(class_weights))

class_weights = {0: 0.00000000000001, 1: 10000000000000}

X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [57]:
for each in y_train:
    if each != 0 and each != 1:
        print(each)

In [58]:
def build_model_2d():
    # Budowa modelu LSTM
    model = Sequential()

    # Warstwa LSTM
    model.add(LSTM(128, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(0.2))  # Dropout dla regularizacji

    # Druga warstwa LSTM
    # model.add(LSTM(128, activation='relu', return_sequences=True))
    # model.add(Dropout(0.2))

    # Trzecia warstwa LSTM
    model.add(LSTM(64, activation='relu'))
    model.add(Dropout(0.2))

    # Warstwa wyjściowa
    model.add(Dense(1, activation='sigmoid'))  # 1 klasa dla klasyfikacji binarnej
    return model

In [59]:
# def build_model_md():
#     # Budowa modelu LSTM
#     model = Sequential()

#     # Warstwa LSTM
#     model.add(LSTM(256, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
#     model.add(Dropout(0.2))  # Dropout dla regularizacji

#     # Druga warstwa LSTM
#     model.add(LSTM(128, activation='relu', return_sequences=True))
#     model.add(Dropout(0.2))

#     # Trzecia warstwa LSTM
#     model.add(LSTM(64, activation='relu'))
#     model.add(Dropout(0.2))

#     # Warstwa wyjściowa
#     model.add(Dense(2, activation='softmax'))  # 2 klasy: benign i attack
#     return model

In [60]:
optimizer = Adam(learning_rate=0.001) 
model = build_model_2d()
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy', 'precision', 'recall'])

# model = build_model_md()
# Kompilacja modelu z optymalizatorem Adam
# model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [61]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10)

# model.fit(X_train, y_train, epochs=5, batch_size=120, validation_split=0.2, callbacks=[early_stopping])
model.fit(X_train, y_train, epochs=5, batch_size=32, class_weight=class_weights, validation_data=(X_test, y_test))

Epoch 1/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step - accuracy: 0.0072 - loss: 2410437632.0000 - precision: 0.0048 - recall: 0.9955 - val_accuracy: 0.0047 - val_loss: 19.0614 - val_precision: 0.0047 - val_recall: 1.0000
Epoch 2/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.0048 - loss: 390.0816 - precision: 0.0048 - recall: 0.9996 - val_accuracy: 0.0047 - val_loss: 27.2618 - val_precision: 0.0047 - val_recall: 1.0000
Epoch 3/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.0047 - loss: 1.3715 - precision: 0.0047 - recall: 0.9994 - val_accuracy: 0.0047 - val_loss: 27.2618 - val_precision: 0.0047 - val_recall: 1.0000
Epoch 4/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.0049 - loss: 13.6956 - precision: 0.0049 - recall: 0.9997 - val_accuracy: 0.0047 - val_loss: 38.6206 - val_precision: 0.0047 - val_recall: 1.0000
Epoch 5/5
11466/11466 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.0050 - loss: 1.9055e-04 - precision: 0.0050 - recall: 1.00

In [62]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

2867/2867 ━━━━━━━━━━━━━━━━━━━━ 1s 369us/step


In [63]:
cm = confusion_matrix(y_test, y_pred_classes)

In [64]:
print(cm)
print(classification_report(y_test, y_pred_classes, target_names=['BENIGN', 'ATTACK']))
# print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

[[91292     0]
 [  434     0]]
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     91292
      ATTACK       0.00      0.00      0.00       434

    accuracy                           1.00     91726
   macro avg       0.50      0.50      0.50     91726
weighted avg       0.99      1.00      0.99     91726

